In [0]:
import csv
import os
import time
import requests

# ============================================================
# Reference data: small CSV files versioned in Git (reference_data/).
# Locations: one point per province (its capital), geocoded with the Open-Meteo geocoding API.
# ============================================================

geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
capital_feature_codes = {"PPLC", "PPLA", "PPLA2"}   # GeoNames: national / regional / provincial capital
location_columns = ["province_code", "province_name", "capital_name", "region_name",
                    "electric_system", "timezone", "latitude", "longitude", "geonames_id"]

# Expected areas per electric system: (min_lat, max_lat, min_lon, max_lon)
electric_system_bounds = {
    "peninsular": (35.9, 43.9, -9.4, 3.4),
    "balearic": (38.6, 40.1, 1.1, 4.4),
    "canary": (27.6, 29.5, -18.2, -13.3),
    "ceuta": (35.8, 35.95, -5.4, -5.2),
    "melilla": (35.2, 35.35, -3.0, -2.85),
}


def find_reference_data_dir(folder_name="reference_data"):
    """Walk up from the running notebook's folder until reference_data/ is found."""
    current = os.getcwd()
    while True:
        candidate = os.path.join(current, folder_name)
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"Folder '{folder_name}' not found from {os.getcwd()} upwards")
        current = parent


def read_reference_csv(file_name):
    with open(os.path.join(find_reference_data_dir(), file_name), encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def write_reference_csv(file_name, rows, fieldnames):
    path = os.path.join(find_reference_data_dir(), file_name)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def geocode_capital(query):
    """Best Spanish capital-type match for a city name. Returns (result, warning)."""
    response = requests.get(
        geocoding_url,
        params={"name": query, "count": 10, "language": "es", "format": "json", "countryCode": "ES"},
        timeout=30,
    )
    response.raise_for_status()
    candidates = response.json().get("results", [])
    capitals = [c for c in candidates if c.get("feature_code") in capital_feature_codes]
    pool = capitals or candidates
    if not pool:
        return None, "no results"
    best = max(pool, key=lambda c: c.get("population") or 0)
    warning = "" if capitals else "no capital-type result, took the most populated match"
    return best, warning


def build_locations(seed_file="provinces_es_seed.csv"):
    """Seed + geocoded coordinates. Returns (locations, review_rows)."""
    locations, review = [], []
    for row in read_reference_csv(seed_file):
        best, warning = geocode_capital(row["geocoding_query"])
        if best is None:
            review.append({**row, "warning": warning})
            continue
        locations.append({
            "province_code": row["province_code"],
            "province_name": row["province_name"],
            "capital_name": row["capital_name"],
            "region_name": row["region_name"],
            "electric_system": row["electric_system"],
            "timezone": row["timezone"],
            "latitude": round(best["latitude"], 4),
            "longitude": round(best["longitude"], 4),
            "geonames_id": best.get("id"),
        })
        review.append({
            "province_code": row["province_code"],
            "capital_name": row["capital_name"],
            "found_name": best.get("name"),
            "found_province": best.get("admin2"),
            "found_region": best.get("admin1"),
            "feature_code": best.get("feature_code"),
            "latitude": round(best["latitude"], 4),
            "longitude": round(best["longitude"], 4),
            "warning": warning,
        })
        time.sleep(0.2)   # be polite with the public API
    return locations, review


def check_locations(locations, expected_count=52):
    """Automatic checks before saving. Returns a list of errors (empty = OK)."""
    errors = []
    if len(locations) != expected_count:
        errors.append(f"expected {expected_count} locations, got {len(locations)}")
    codes = [l["province_code"] for l in locations]
    if len(codes) != len(set(codes)):
        errors.append("duplicated province_code")
    for l in locations:
        bounds = electric_system_bounds.get(l["electric_system"])
        if bounds is None:
            errors.append(f"{l['province_code']}: unknown electric_system {l['electric_system']}")
            continue
        min_lat, max_lat, min_lon, max_lon = bounds
        if not (min_lat <= l["latitude"] <= max_lat and min_lon <= l["longitude"] <= max_lon):
            errors.append(f"{l['province_code']} {l['capital_name']}: ({l['latitude']}, {l['longitude']}) "
                          f"outside the {l['electric_system']} area")
    return errors